# 🎓 Educational Tutorial: EfficientNet-B0

## 1. History & Original Paper
* **Paper**: *EfficientNet: Rethinking Model Scaling for Convolutional Neural Networks* (Tan & Le, 2019)
* **Concept**: Compound scaling of width, depth, and resolution simultaneously. MBConv with Squeeze-and-Excitation blocks.

## 2. Why ResNet was Insufficient
Traditional scaling approaches typically scaled only one dimension (e.g. depth in ResNet). This led to saturated gains. EfficientNet scales depth, width, and input resolution uniformly using a fixed compound coefficient, yielding better accuracy and efficiency.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
import pandas as pd
from PIL import Image
from pathlib import Path
import time
import math

print("Imports complete.")

In [ ]:
class Swish(nn.Module):
    def forward(self, x):
        return x * torch.sigmoid(x)

class SqueezeExcitation(nn.Module):
    def __init__(self, in_channels, reduced_dim):
        super().__init__()
        self.se = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Conv2d(in_channels, reduced_dim, 1),
            Swish(),
            nn.Conv2d(reduced_dim, in_channels, 1),
            nn.Sigmoid()
        )
    def forward(self, x):
        return x * self.se(x)

class MBConv(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size, stride, expand_ratio, se_ratio=0.25):
        super().__init__()
        self.stride = stride
        self.use_residual = (self.stride == 1 and in_channels == out_channels)
        expanded_channels = in_channels * expand_ratio
        
        layers = []
        if expand_ratio != 1:
            layers.extend([
                nn.Conv2d(in_channels, expanded_channels, 1, bias=False),
                nn.BatchNorm2d(expanded_channels),
                Swish()
            ])
        padding = (kernel_size - 1) // 2
        layers.extend([
            nn.Conv2d(expanded_channels, expanded_channels, kernel_size, stride, padding, groups=expanded_channels, bias=False),
            nn.BatchNorm2d(expanded_channels),
            Swish()
        ])
        reduced_dim = max(1, int(in_channels * se_ratio))
        layers.append(SqueezeExcitation(expanded_channels, reduced_dim))
        layers.extend([
            nn.Conv2d(expanded_channels, out_channels, 1, bias=False),
            nn.BatchNorm2d(out_channels)
        ])
        self.block = nn.Sequential(*layers)
    def forward(self, x):
        if self.use_residual:
            return x + self.block(x)
        return self.block(x)

In [ ]:
class EfficientNetB0FromScratch(nn.Module):
    def __init__(self, in_channels=3, num_classes=10):
        super().__init__()
        self.stem = nn.Sequential(
            nn.Conv2d(in_channels, 32, 3, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(32),
            Swish()
        )
        b0_config = [
            (1, 16, 1, 1, 3),
            (6, 24, 2, 2, 3),
            (6, 40, 2, 2, 5),
            (6, 80, 3, 2, 3),
            (6, 112, 3, 1, 5),
            (6, 192, 4, 2, 5),
            (6, 320, 1, 1, 3)
        ]
        blocks = []
        in_c = 32
        for expand_ratio, out_c, num_layers, stride, kernel_size in b0_config:
            for i in range(num_layers):
                s = stride if i == 0 else 1
                blocks.append(MBConv(in_c, out_c, kernel_size, s, expand_ratio))
                in_c = out_c
        self.blocks = nn.Sequential(*blocks)
        self.head = nn.Sequential(
            nn.Conv2d(in_c, 1280, 1, bias=False),
            nn.BatchNorm2d(1280),
            Swish(),
            nn.AdaptiveAvgPool2d(1)
        )
        self.classifier = nn.Sequential(
            nn.Dropout(p=0.2),
            nn.Linear(1280, num_classes)
        )
    def forward(self, x):
        x = self.stem(x)
        x = self.blocks(x)
        x = self.head(x)
        x = torch.flatten(x, start_dim=1)
        return self.classifier(x)

model = EfficientNetB0FromScratch()
print("EfficientNet-B0 Parameter Count:", sum(p.numel() for p in model.parameters()))

In [ ]:
class NotebookEuroSATDataset(Dataset):
    def __init__(self, csv_file, root_dir, transform=None):
        self.data = pd.read_csv(csv_file)
        self.root_dir = Path(root_dir)
        self.transform = transform
        
    def __len__(self):
        return len(self.data)
        
    def __getitem__(self, idx):
        row = self.data.iloc[idx]
        img_path = self.root_dir / row["image_path"]
        image = Image.open(img_path).convert("RGB")
        label = row["label"]
        
        if self.transform:
            image = self.transform(image)
            
        return {
            "image": image,
            "label": label
        }

In [ ]:
PROJECT_ROOT = Path("../../")
csv_path = PROJECT_ROOT / "data/processed/train.csv"
val_csv_path = PROJECT_ROOT / "data/processed/validation.csv"
img_dir = PROJECT_ROOT / "data/raw/EuroSAT_RGB"

if not csv_path.exists():
    PROJECT_ROOT = Path(".").resolve().parents[1]
    csv_path = PROJECT_ROOT / "data/processed/train.csv"
    val_csv_path = PROJECT_ROOT / "data/processed/validation.csv"
    img_dir = PROJECT_ROOT / "data/raw/EuroSAT_RGB"

train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

if csv_path.exists() and img_dir.exists():
    train_dataset = NotebookEuroSATDataset(csv_path, img_dir, train_transform)
    val_dataset = NotebookEuroSATDataset(val_csv_path, img_dir, val_transform)
    
    train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True, num_workers=2)
    val_loader = DataLoader(val_dataset, batch_size=16, shuffle=False, num_workers=2)
    print("EuroSAT dataset splits loaded successfully.")
else:
    print("Warning: EuroSAT files not found! Mocking data loading.")
    class DummyDataset(Dataset):
        def __len__(self): return 16
        def __getitem__(self, idx): return {"image": torch.randn(3, 224, 224), "label": torch.randint(0, 10, (1,)).item()}
    train_loader = DataLoader(DummyDataset(), batch_size=4, shuffle=True)
    val_loader = DataLoader(DummyDataset(), batch_size=4, shuffle=False)

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Training device:", device)

model = EfficientNetB0FromScratch().to(device)
optimizer = optim.Adam(model.parameters(), lr=1e-4)
criterion = nn.CrossEntropyLoss()

model.train()
start_time = time.time()
epoch_loss = 0.0
correct = 0
total = 0

print("Starting training verification step...")
max_batches = 2
for idx, batch in enumerate(train_loader):
    if idx >= max_batches: break
    images = batch["image"].to(device)
    labels = batch["label"].to(device)
    
    optimizer.zero_grad()
    outputs = model(images)
    loss = criterion(outputs, labels)
    loss.backward()
    optimizer.step()
    
    epoch_loss += loss.item() * images.size(0)
    correct += (outputs.argmax(dim=1) == labels).sum().item()
    total += labels.size(0)
    
print(f"Verification training step complete in {time.time() - start_time:.2f}s | Loss: {epoch_loss/total:.4f} | Acc: {correct/total*100:.2f}%")